# 02 — Dataset Viewer

**Goal:** Visually audit image/mask pairs before training.  
Confirm that the masks line up with the rover images and that class colours are what you expect.

---

## Why do we visualise before training?

This is a **data audit** — one of the most important steps in any ML project.

Things to look for:
- Do the mask boundaries follow visible terrain edges in the photo?
- Are the class colours consistent across samples?
- Do any masks look completely black (all zeros) or corrupted?
- Are there many 255-labelled (ignored) pixels that reduce usable training data?

> **Prerequisite:** Run `01_dataset_inspection.ipynb` first to build and validate `pairs`.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from src.data_paths import RAW_DATA_DIR
from src.dataset import find_image_files, find_mask_files, build_pairs_by_stem
from src.visualize import show_sample, show_image_mask_overlay, print_mask_distribution, CLASS_NAMES

## Step 1 — Build Pairs

If you have already confirmed correct pairing in notebook 01, re-run it here.  
Alternatively, you can define pairs manually for a quick test.

In [ ]:
DATA_ROOT = RAW_DATA_DIR  # adjust if needed

image_files = find_image_files(DATA_ROOT)
mask_files  = find_mask_files(DATA_ROOT)
pairs       = build_pairs_by_stem(image_files, mask_files)

print(f"Total pairs available: {len(pairs)}")

# ------------------------------------------------------------------
# Manual override: uncomment and edit to define a few pairs by hand
# ------------------------------------------------------------------
# pairs = [
#     (DATA_ROOT / "images" / "some_image.jpg", DATA_ROOT / "masks" / "some_image.png"),
# ]

## Step 2 — View Individual Samples

`show_sample` loads the files, prints diagnostics, and shows three panels:  
original image / class mask / overlay.

In [ ]:
# Visualise the first sample
if pairs:
    show_sample(*pairs[0])
else:
    print("No pairs found. Check DATA_ROOT and run 01_dataset_inspection.ipynb first.")

## Step 3 — View Multiple Samples

Browse through up to 25 samples to get a broad sense of the data quality.

In [ ]:
# Adjust NUM_SAMPLES to see more or fewer
NUM_SAMPLES = 5  # start small; increase to 25 after verifying everything looks correct

for i, (img_path, msk_path) in enumerate(pairs[:NUM_SAMPLES]):
    print(f"\n{'='*60}")
    print(f"Sample {i+1}/{NUM_SAMPLES}")
    show_sample(img_path, msk_path)

## Step 4 — Class Distribution Across All Viewed Samples

Understanding class imbalance is important for choosing a loss function and  
interpreting metric results.

In [ ]:
import numpy as np
from PIL import Image
from collections import defaultdict

# Count pixel totals across the first NUM_SAMPLES pairs
class_totals = defaultdict(int)

for img_path, msk_path in pairs[:NUM_SAMPLES]:
    mask = np.array(Image.open(msk_path))
    for cid in np.unique(mask):
        class_totals[int(cid)] += int((mask == cid).sum())

grand_total = sum(class_totals.values())
print(f"Aggregate class distribution across {NUM_SAMPLES} samples ({grand_total} total pixels):")
print(f"\n{'Class':>5}  {'Name':>12}  {'Pixels':>10}  {'%':>6}")
print("-" * 42)
for cid in sorted(class_totals):
    name = CLASS_NAMES.get(cid, "unknown")
    count = class_totals[cid]
    pct = 100.0 * count / grand_total
    print(f"{cid:>5}  {name:>12}  {count:>10}  {pct:>5.1f}%")

## Next Steps

Once you are satisfied that:
- Mask boundaries align with visible terrain in the images.
- Class IDs are correct and the distribution is understood.
- There are no obviously corrupted samples.

Open `03_baseline_training.ipynb` to train your first model.